# Starter Notebook

Install and import required libraries

In [ ]:
!pip install datasets evaluate peft trl bitsandbytes -q
!pip install nvidia-ml-py3 -q

In [ ]:
# Optional experiment tracking (Weights & Biases).
# Set WANDB_API_KEY in your environment to enable; safe to skip.
import os
try:
    import wandb
    if os.environ.get("WANDB_API_KEY"):
        wandb.login(key=os.environ["WANDB_API_KEY"])
except Exception as e:
    print("wandb disabled:", e)


In [ ]:
import os
import pandas as pd
import torch
from transformers import RobertaModel, RobertaTokenizer, TrainingArguments, Trainer, DataCollatorWithPadding, RobertaForSequenceClassification
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset, Dataset, ClassLabel
import pickle
import string
import re

## Load Tokenizer and Preprocess Data

In [ ]:
base_model = 'roberta-base'

dataset_train = load_dataset('ag_news', split='train')
dataset_test = load_dataset('ag_news', split='test')

tokenizer = RobertaTokenizer.from_pretrained(base_model)

def preprocess(examples):
    # Punctuation to keep
    punctuation_to_keep = ".,'-$%?!"

    # Remove unwanted punctuation
    processed_texts = [
        ''.join(
            char for char in text
            if char not in string.punctuation or char in punctuation_to_keep
        )
        for text in examples["text"]
    ]

    # Remove HTML tags, URLs, and extra whitespace
    processed_texts = [re.sub(r'<.*?>', '', text) for text in processed_texts]
    processed_texts = [re.sub(r'http\S+', '', text) for text in processed_texts]
    processed_texts = [re.sub(r'\s+', ' ', text).strip() for text in processed_texts]

    # Tokenize the processed texts
    tokenized = tokenizer(processed_texts, truncation=True)
    return tokenized

train_dataset = dataset_train.map(preprocess, batched=True,  remove_columns=["text"])
train_dataset = train_dataset.rename_column("label", "labels")

eval_dataset = dataset_test.map(preprocess, batched=True,  remove_columns=["text"])
eval_dataset = eval_dataset.rename_column("label", "labels")

In [ ]:
# Extract the number of classess and their names
num_labels = train_dataset.features['labels'].num_classes
class_names = train_dataset.features["labels"].names
print(f"number of labels: {num_labels}")
print(f"the labels: {class_names}")

# Create an id2label mapping
# We will need this for our classifier.
id2label = {i: label for i, label in enumerate(class_names)}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True, return_tensors="pt")

## Load Pre-trained Model
Set up config for pretrained model and download it from hugging face

In [ ]:
model = RobertaForSequenceClassification.from_pretrained(base_model, id2label=id2label)

## Setup LoRA Config
Setup PEFT config and get peft model for finetuning

In [ ]:
# PEFT Config
peft_config = LoraConfig(
    r=11,
    lora_alpha=22,
    lora_dropout=0.1,
    bias = 'none',
    target_modules = ['query', 'value'],
    task_type="SEQ_CLS",
)

In [ ]:
peft_model = get_peft_model(model, peft_config)

In [ ]:
print('PEFT Model')
peft_model.print_trainable_parameters()

## Training Setup

In [ ]:
# To track evaluation accuracy during training
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    # Calculate accuracy
    accuracy = accuracy_score(labels, preds)

    return {'accuracy': accuracy}

In [ ]:
# Setup Training args
output_dir = "results"
training_args = TrainingArguments(

    output_dir=output_dir,
    report_to=None,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=5,

    learning_rate=2e-4,
    num_train_epochs=5,
    # max_steps=1000,
    warmup_ratio=0.06,
    weight_decay=0.01,
    label_smoothing_factor=0.2,
    seed=42,

    dataloader_num_workers=2,
    dataloader_prefetch_factor=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    optim="adamw_torch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,
    dataloader_pin_memory=True,
    dataloader_persistent_workers=True,
    neftune_noise_alpha=5,
)

def get_trainer(model):
      return Trainer(
          model=model,
          args=training_args,
          compute_metrics=compute_metrics,
          train_dataset=train_dataset,
          eval_dataset=eval_dataset,
          data_collator=data_collator,
      )

### Start Training

In [ ]:
peft_lora_finetuning_trainer = get_trainer(peft_model)
result = peft_lora_finetuning_trainer.train()

## Evaluate Finetuned Model


### Run Inference on eval_dataset

In [ ]:
from torch.utils.data import DataLoader
import evaluate
from tqdm import tqdm

def evaluate_model(inference_model, dataset, labelled=True, batch_size=8, data_collator=None):
    """
    Evaluate a PEFT model on a dataset.

    Args:
        inference_model: The model to evaluate.
        dataset: The dataset (Hugging Face Dataset) to run inference on.
        labelled (bool): If True, the dataset includes labels and metrics will be computed.
                         If False, only predictions will be returned.
        batch_size (int): Batch size for inference.
        data_collator: Function to collate batches. If None, the default collate_fn is used.

    Returns:
        If labelled is True, returns a tuple (metrics, predictions)
        If labelled is False, returns the predictions.
    """
    # Create the DataLoader
    eval_dataloader = DataLoader(dataset, batch_size=batch_size, collate_fn=data_collator)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    inference_model.to(device)
    inference_model.eval()

    all_predictions = []
    if labelled:
        metric = evaluate.load('accuracy')

    # Loop over the DataLoader
    for batch in tqdm(eval_dataloader):
        # Move each tensor in the batch to the device
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = inference_model(**batch)
        predictions = outputs.logits.argmax(dim=-1)
        all_predictions.append(predictions.cpu())

        if labelled:
            # Expecting that labels are provided under the "labels" key.
            references = batch["labels"]
            metric.add_batch(
                predictions=predictions.cpu().numpy(),
                references=references.cpu().numpy()
            )

    # Concatenate predictions from all batches
    all_predictions = torch.cat(all_predictions, dim=0)

    if labelled:
        eval_metric = metric.compute()
        print("Evaluation Metric:", eval_metric)
        return eval_metric, all_predictions
    else:
        return all_predictions

In [ ]:
# Check evaluation accuracy
_, _ = evaluate_model(peft_model, eval_dataset, True, 1, data_collator)

### Run Inference on unlabelled dataset

In [ ]:
# Load your unlabelled data
unlabelled_dataset = pd.read_pickle("/kaggle/input/deep-learning-spring-2025-project-2/test_unlabelled.pkl")
test_dataset = unlabelled_dataset.map(preprocess, batched=True, remove_columns=["text"])
unlabelled_dataset

In [ ]:
# Run inference and save predictions
preds = evaluate_model(peft_model, test_dataset, False, 1, data_collator)
df_output = pd.DataFrame({
    'ID': range(len(preds)),
    'Label': preds.numpy()  # or preds.tolist()
})
df_output.to_csv(os.path.join(output_dir,"inference_output.csv"), index=False)
print("Inference complete. Predictions saved to inference_output.csv")